# 랭체인(LangChain) MultiVector Retriever 예제
## 작성자 : AISchool ( http://aischool.ai/%ec%98%a8%eb%9d%bc%ec%9d%b8-%ea%b0%95%ec%9d%98-%ec%b9%b4%ed%85%8c%ea%b3%a0%eb%a6%ac/ )
## Reference : https://python.langchain.com/v0.1/docs/modules/data_connection/retrievers/multi_vector/

**문서당 여러 벡터를 저장하는 것**은 종종 유익할 수 있습니다. 여러 가지 사용 사례에서 이것이 유익합니다. LangChain에는 이러한 설정을 쉽게 쿼리할 수 있는 기본 **MultiVectorRetriever**가 있습니다. 문서당 여러 벡터를 생성하는 방법에는 여러가지 복잡성들이 존재합니다. 이번 예제에서는 이러한 벡터를 생성하고 MultiVectorRetriever를 사용하는 일반적인 방법을 다룹니다.








문서당 여러 벡터를 생성하는 방법에는 다음이 포함됩니다:

*   **더 작은 청크(Smaller chunks)**: 문서를 더 작은 청크로 나누고, 그것들을 임베딩합니다. 가장 마지막 레이어 (이것은 ParentDocumentRetriever입니다).
*   **요약(Summary)**: 각 문서에 대한 요약을 생성하고, 그것을 문서와 함께(또는 문서 대신) 임베딩합니다, 중간 레이어
*   **가상 질문(Hypothetical questions)**: 각 문서가 적절하게 답변할 수 있는 가상 질문(hypothetical questions)을 생성하고, 그것들을 문서와 함께(또는 문서 대신) 임베딩합니다.


이 방법은 임베딩을 수동으로 추가하는 또 다른 방법을 가능하게 한다는 점을 주목하세요. 이는 특정 문서가 검색되도록 해야 하는 질문이나 쿼리를 명시적으로 추가할 수 있기 때문에 매우 유용하며, 더 많은 제어권을 제공합니다.

# LangChain 라이브러리 설치

In [ ]:
# 2026년 최신 RAG 실습을 위한 전체 라이브러리 설치
!pip install -U langchain langchain-openai langchain-community langchain-chroma \
langchain-classic langchain-text-splitters langchain-classic\
openai chromadb tiktoken pypdf unstructured sentence-transformers

# OpenAI API Key 설정

In [ ]:
OPENAI_KEY = "Input Your Key"

In [ ]:
from langchain_classic.retrievers import MultiVectorRetriever
from langchain_core.stores import InMemoryStore
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
loaders = [
    PyPDFLoader("https://snuac.snu.ac.kr/2015_snuac/wp-content/uploads/2015/07/asiabrief_3-26.pdf")
]
docs = []
for loader in loaders:
    docs.extend(loader.load())
text_splitter = RecursiveCharacterTextSplitter(chunk_size=10000)
docs = text_splitter.split_documents(docs)

In [ ]:
docs

In [ ]:
len(docs)

# Smaller chunks

종종 **더 큰 정보 청크를 검색하되, 더 작은 청크를 임베딩하는 것이 유용**할 수 있습니다. 이렇게 하면 임베딩이 의미(Semantic meaning)를 가능한 한 정확하게 포착할 수 있지만, 가능한 많은 컨텍스트를 하위 처리 단계로 전달할 수 있습니다. 이것이 바로 ParentDocumentRetriever가 하는 일입니다. 여기서는 이 과정의 내부 동작을 설명합니다.

In [ ]:
vectorstore = Chroma(
    collection_name="full_documents",
    embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_KEY)
)

store = InMemoryStore()
id_key = "doc_id" #부모청크와 자식 청크간의 이어주는 키

retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    docstore=store,
    id_key=id_key,
)

In [ ]:
import uuid

doc_ids = [str(uuid.uuid4()) for _ in docs]
doc_ids

In [ ]:
# The splitter to use to create smaller chunks
child_text_splitter = RecursiveCharacterTextSplitter(chunk_size=400)

In [ ]:
sub_docs = []
for i, doc in enumerate(docs):
    _id = doc_ids[i]
    _sub_docs = child_text_splitter.split_documents([doc])
    for _doc in _sub_docs:
        _doc.metadata[id_key] = _id
    sub_docs.extend(_sub_docs)

In [ ]:
sub_docs

In [ ]:
len(sub_docs)

In [ ]:
retriever.vectorstore.add_documents(sub_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

In [ ]:
# Vectorstore alone retrieves the small chunks
#Child Chunk
retriever.vectorstore.similarity_search("한국 저출산의 원인이 뭐야?")[0]

In [ ]:
# Retriever returns larger chunks
#Parent Chunk
larger_chunks = retriever.invoke("한국 저출산의 원인이 뭐야?")[0].page_content
larger_chunks

In [ ]:
len(larger_chunks)

# Summary

요약(summary)은 청크의 내용을 더 정확하게 요약하여 더 나은 검색 결과를 얻을 수 있습니다. 여기에서는 요약을 생성하고 이를 임베딩하는 방법을 보여줍니다.

In [ ]:
import uuid

from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

In [ ]:
chain = (
    {"doc": lambda x: x.page_content}
    | ChatPromptTemplate.from_template("아래 문서를 요약하세요.:\n\n{doc}")
    | ChatOpenAI(max_retries=0, openai_api_key=OPENAI_KEY)
    | StrOutputParser()
)

In [ ]:
docs

In [ ]:
summaries = chain.batch(docs, {"max_concurrency": 5})
summaries

In [ ]:
len(summaries)

In [ ]:
# The vectorstore to use to index the child chunks
vectorstore = Chroma(collection_name="summaries", embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_KEY))
# The storage layer for the parent documents
store = InMemoryStore()
id_key = "doc_id"
# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)

In [ ]:
doc_ids = [str(uuid.uuid4()) for _ in docs]
doc_ids

In [ ]:
summary_docs = [
    Document(page_content=s, metadata={id_key: doc_ids[i]})
    for i, s in enumerate(summaries)
]

In [ ]:
summary_docs

In [ ]:
retriever.vectorstore.add_documents(summary_docs) #하위
retriever.docstore.mset(list(zip(doc_ids, docs))) #상위

In [ ]:
# # We can also add the original chunks to the vectorstore if we so want
# for i, doc in enumerate(docs):
#     doc.metadata[id_key] = doc_ids[i]
# retriever.vectorstore.add_documents(docs)

In [ ]:
sub_docs = vectorstore.similarity_search("한국 저출산의 원인이 뭐야?")
sub_docs

In [ ]:
sub_docs[0]

In [ ]:
larger_chunks = retriever.invoke("한국 저출산의 원인이 뭐야?")
larger_chunks

In [ ]:
len(larger_chunks[0].page_content)

# Hypothetical Queries

LLM을 사용하여 특정 문서에 대해 물어볼 수 있는 **가상 질문 목록(hypothetical questions)**을 생성할 수도 있습니다. 이러한 질문들은 임베딩될 수 있습니다.

In [ ]:
#Function Calling
functions = [
    {
        "name": "hypothetical_questions",
        "description": "Generate hypothetical questions",
        "parameters": {
            "type": "object",
            "properties": {
                "questions": {
                    "type": "array",
                    "items": {"type": "string"},
                },
            },
            "required": ["questions"],
        },
    }
]

In [ ]:
from langchain_core.output_parsers.openai_functions import JsonKeyOutputFunctionsParser

chain = (
    {"doc": lambda x: x.page_content}
    # Only asking for 3 hypothetical questions, but this could be adjusted
    | ChatPromptTemplate.from_template(
        "다음 문서가 답할 수 있는 가상 질문 3개를 생성하십시오:\n\n{doc}"
    )
    | ChatOpenAI(max_retries=0, model="gpt-4o-mini", openai_api_key=OPENAI_KEY).bind(
        functions=functions, function_call={"name": "hypothetical_questions"}
    )
    | JsonKeyOutputFunctionsParser(key_name="questions")
)

In [ ]:
docs[0]

In [ ]:
chain.invoke(docs[0])

In [ ]:
docs

In [ ]:
len(docs)

In [ ]:
hypothetical_questions = chain.batch(docs, {"max_concurrency": 1})
hypothetical_questions

In [ ]:
# The vectorstore to use to index the child chunks
vectorstore = Chroma(
    collection_name="hypo-questions", embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_KEY)
)
# The storage layer for the parent documents
store = InMemoryStore()
id_key = "doc_id"
# The retriever (empty to start)
retriever = MultiVectorRetriever(
    vectorstore=vectorstore,
    byte_store=store,
    id_key=id_key,
)

In [ ]:
doc_ids = [str(uuid.uuid4()) for _ in docs]
doc_ids

In [ ]:
docs[3]

In [ ]:
question_docs = []
for i, question_list in enumerate(hypothetical_questions):
    question_docs.extend(
        [Document(page_content=s, metadata={id_key: doc_ids[i]}) for s in question_list]
    )

In [ ]:
question_docs

In [ ]:
retriever.vectorstore.add_documents(question_docs)
retriever.docstore.mset(list(zip(doc_ids, docs)))

In [ ]:
sub_docs = vectorstore.similarity_search("한국 저출산의 원인이 뭐야?")

In [ ]:
sub_docs

In [ ]:
larger_chunks = retriever.invoke("한국 저출산의 원인이 뭐야?")
larger_chunks

In [ ]:
len(larger_chunks[0].page_content)

In [ ]:
sub_docs = vectorstore.similarity_search("저출산으로 인한 노동력 부족에 대응하기 위해 어떤 전략들이 필요해?")
sub_docs

In [ ]:
larger_chunks = retriever.invoke("저출산으로 인한 노동력 부족에 대응하기 위해 어떤 전략들이 필요해?")
larger_chunks[0].page_content